# Toxic Comment Moderation: TF-IDF baseline vs. SOTA LLMs vs. Jev

**Question:** a moderation queue gets thousands of comments an hour. Which model flags the toxic ones best, how many decisions can it make on its own without a human, and what does each one cost in time and money?

**Contenders:** a TF-IDF + logistic regression baseline trained on 160k labelled comments, three mid-tier frontier LLMs (Claude Sonnet 5, GPT-6 Sol, Gemini 3.8 Flash), and Jev, TypeSafe's calibrated decision model.

This is the use case Jev is built for: short text in, several narrow yes/no judgements out, at high volume, where a trustworthy probability decides what gets automated and what goes to a person. The attrition bench asked Jev to do statistics over a CSV table, which is its weakest mode. This one plays to its strengths on purpose, and says so.

**The rules every model plays by**

| Rule | Why |
|---|---|
| Six labels per comment, one call per comment | Jigsaw's labels: toxic, severe_toxic, obscene, threat, insult, identity_hate. Jev gets six yes/no questions in one call; the LLMs return all six in one JSON answer. Each model gets its native mode. |
| Identical label definitions for everyone | The same one-line definition per label goes into the LLM prompt and into Jev's question criteria. |
| No tools, provider-default reasoning | The honest out-of-the-box baseline. Nothing is tuned per model. |
| Every model gives a probability per label | Jev returns calibrated probabilities; the LLMs are asked for 0-100 per label. The baseline uses `predict_proba`. So every model can be scored threshold-free. |
| Headline metrics: **macro ROC AUC**, **calibration error** and **automation coverage** | AUC measures ranking. Calibration says whether a 0.9 means 90%. Coverage asks what share of comments a model can decide alone at 98% accuracy, with the rest going to a human. |
| Cost and latency at production scale | Every call is timed and priced, then extrapolated to 1M comments. An optional parallel run measures throughput. |

**Sections:** 1 Setup, 2 Test set, 3 Baseline, 4 Scorer, 5 Prompt, 6 Models, 7 Run, 8 Results, 9 Throughput, then caveats and the figures.

## 1. Setup and data

The [Jigsaw Toxic Comment Classification Challenge](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) data comes straight from Kaggle through `kagglehub` and is cached outside the repo, so no comment text is ever committed. It's a competition dataset, so accept the competition rules on Kaggle once with the account whose API token you use. One seed drives every random step.

In [ ]:
%pip install -q kagglehub pandas scikit-learn matplotlib typesafe-sdk anthropic openai google-genai

import random
from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Pull straight from Kaggle; cached under ~/.cache/kagglehub, never lands in the repo
KAGGLE_COMPETITION = "jigsaw-toxic-comment-classification-challenge"
data_dir = Path(kagglehub.competition_download(KAGGLE_COMPETITION))

def read_split(name):
    # Kaggle ships each file as name.csv.zip; pandas reads the zip directly. keep_default_na: a comment
    # that says "NA" or "null" must stay text.
    return pd.read_csv(next(data_dir.rglob(f"{name}.csv*")), keep_default_na=False)

train_comments = read_split("train")
test_comments = read_split("test").merge(read_split("test_labels"), on="id")
# Test rows labelled -1 were never scored in the competition, so they have no ground truth
test_comments = test_comments[(test_comments[LABELS] >= 0).all(axis=1)].reset_index(drop=True)
for frame in (train_comments, test_comments):
    frame[LABELS] = frame[LABELS].astype(int)

print(f"train n={len(train_comments):,}   test (labelled) n={len(test_comments):,}")
display(pd.DataFrame({"train rate": train_comments[LABELS].mean(), "test rate": test_comments[LABELS].mean()}).style.format("{:.2%}"))

## 2. The test set

At natural rates a 1,000-comment sample would hold about 3 threats, too few to score. So the test set is **enriched**: half clean comments (no label at all) and half flagged comments, with at least 40 positives guaranteed for each rare label. Every model faces the same enriched mix; precision and coverage are about this mix, not a live feed (see caveats).

Test comments come from Jigsaw's **test** split, which the baseline never trains on. Exact duplicates of a training comment are dropped. Only the comment **ids** are frozen to `test_comment_ids.csv`, never the text.

In [ ]:
N_TEST = 1_000                  # total comments; every LLM call is one comment, so cost scales with this
MIN_POSITIVES_PER_LABEL = 40    # rare labels (threat, identity_hate, severe_toxic) get at least this many
TEST_IDS_PATH = Path("test_comment_ids.csv")

# No test comment may appear verbatim in the baseline's training data
train_texts = set(train_comments["comment_text"].str.strip())
candidates = test_comments[~test_comments["comment_text"].str.strip().isin(train_texts)]
candidates = candidates[candidates["comment_text"].str.strip() != ""]
print(f"dropped {len(test_comments) - len(candidates)} test comments that are empty or duplicate a training comment")

def draw_test_set(candidates, rng):
    is_flagged = candidates[LABELS].any(axis=1)
    clean_pool, flagged_pool = candidates[~is_flagged], candidates[is_flagged]
    n_flagged = N_TEST // 2

    chosen = set()
    # Rarest label first, so its guaranteed positives are picked before common labels fill the quota
    for label in sorted(LABELS, key=lambda label: flagged_pool[label].sum()):
        already = flagged_pool.index.isin(list(chosen)) & (flagged_pool[label] == 1)
        needed = MIN_POSITIVES_PER_LABEL - already.sum()
        pool = flagged_pool[(flagged_pool[label] == 1) & ~flagged_pool.index.isin(list(chosen))]
        if needed > 0:
            chosen.update(rng.choice(pool.index, size=min(needed, len(pool)), replace=False))
    rest = flagged_pool.index[~flagged_pool.index.isin(list(chosen))]
    chosen.update(rng.choice(rest, size=n_flagged - len(chosen), replace=False))
    chosen.update(rng.choice(clean_pool.index, size=N_TEST - n_flagged, replace=False))
    return candidates.loc[sorted(chosen)].sample(frac=1, random_state=SEED)   # shuffled: no label order in the run

if TEST_IDS_PATH.exists():
    frozen_ids = pd.read_csv(TEST_IDS_PATH)["id"]
    test_set = candidates.set_index("id").loc[frozen_ids].reset_index()
    print(f"loaded the frozen test set from {TEST_IDS_PATH}")
else:
    test_set = draw_test_set(candidates, np.random.default_rng(SEED)).reset_index(drop=True)
    test_set[["id"]].to_csv(TEST_IDS_PATH, index=False)
    print(f"drew a new test set and froze its ids to {TEST_IDS_PATH}")

assert len(test_set) == N_TEST, f"frozen test set has {len(test_set)} comments, N_TEST is {N_TEST}; delete {TEST_IDS_PATH} to redraw"
y_true = test_set[LABELS].to_numpy()
print(f"test set n={len(test_set):,}   flagged={test_set[LABELS].any(axis=1).sum()}   clean={(~test_set[LABELS].any(axis=1)).sum()}")
print(f"median length {test_set['comment_text'].str.len().median():.0f} chars, p95 {test_set['comment_text'].str.len().quantile(0.95):.0f}")
test_set[LABELS].sum().rename("positives").to_frame().T

## 3. The baseline: TF-IDF + logistic regression

The classic Jigsaw baseline: word 1-2 grams plus character 3-5 grams, one logistic regression per label, trained on all 160k labelled training comments. This is what you get if you already have a big labelled dataset and a data scientist. The LLMs and Jev get **zero** training examples, so the baseline is a high bar on accuracy, not a like-for-like competitor.

In [ ]:
import time

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

started = time.perf_counter()
word_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=100_000, sublinear_tf=True, strip_accents="unicode", min_df=2)
char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=100_000, sublinear_tf=True, min_df=3)
train_features = hstack([word_vectorizer.fit_transform(train_comments["comment_text"]),
                         char_vectorizer.fit_transform(train_comments["comment_text"])]).tocsr()
test_features = hstack([word_vectorizer.transform(test_set["comment_text"]),
                        char_vectorizer.transform(test_set["comment_text"])]).tocsr()

baseline_probabilities = np.zeros((len(test_set), len(LABELS)))
for column, label in enumerate(LABELS):
    classifier = LogisticRegression(C=4.0, solver="liblinear", random_state=SEED)
    classifier.fit(train_features, train_comments[label])
    baseline_probabilities[:, column] = classifier.predict_proba(test_features)[:, 1]
baseline_seconds = time.perf_counter() - started

started = time.perf_counter()
_ = classifier.predict_proba(test_features)
baseline_latency_per_comment = (time.perf_counter() - started) / len(test_set) * len(LABELS)
print(f"trained 6 classifiers on {train_features.shape[0]:,} comments x {train_features.shape[1]:,} features in {baseline_seconds:.0f} s")
print(f"inference: {baseline_latency_per_comment * 1000:.3f} ms per comment (all six labels, on this machine)")

## 4. One scorer for every model

`score_probabilities` scores a model from its probability matrix (comments x six labels):

- **ROC AUC** per label and macro-averaged, with a bootstrap 95% CI on the macro. Every model is scored on the **same** resamples, so intervals are directly comparable.
- **F1, recall, precision** at a 0.5 threshold, per label.
- **ECE** (expected calibration error, 10 equal-width bins): the average gap between a stated probability and the observed rate. 0 is perfect.
- **Brier score**: mean squared error of the probabilities.
- **Automation coverage** on the `toxic` label: sort comments by how sure the model is, then take the largest confident share whose decisions are still at least 98% correct. That share is decided automatically; the rest goes to a human. Ties are kept together, so a model that says "100" for everything can't split a group to look better.

A missing or unparseable answer becomes 0.5 for every label: it ranks as unknown, can never be auto-decided, and always goes to a human.

In [ ]:
from sklearn.metrics import roc_auc_score

BOOTSTRAP_RESAMPLES = 1000
AUTOMATION_TARGET_ACCURACY = 0.98
THRESHOLD = 0.5
CALIBRATION_BINS = 10
bootstrap_index = np.random.default_rng(SEED).integers(0, N_TEST, size=(BOOTSTRAP_RESAMPLES, N_TEST))

def expected_calibration_error(truth, probabilities, bins=CALIBRATION_BINS):
    bin_of = np.minimum((probabilities * bins).astype(int), bins - 1)
    gap = 0.0
    for bin_number in range(bins):
        in_bin = bin_of == bin_number
        if in_bin.any():
            gap += in_bin.mean() * abs(probabilities[in_bin].mean() - truth[in_bin].mean())
    return gap

def automation_coverage(truth, probabilities, target=AUTOMATION_TARGET_ACCURACY):
    # Decide the most confident comments automatically. Only cut between distinct confidence values, so a block of
    # tied answers is automated whole or not at all. Returns (share automated, accuracy on that share).
    decisions = probabilities >= THRESHOLD
    confidence = np.abs(probabilities - 0.5)
    best = (0.0, np.nan)
    for cutoff in np.unique(confidence)[::-1]:
        if cutoff == 0:   # exactly 0.5 = no answer; never automated
            break
        automated = confidence >= cutoff
        accuracy = (decisions[automated] == truth[automated]).mean()
        if accuracy >= target:
            best = (automated.mean(), accuracy)
    return best

def score_probabilities(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    scores = {}
    for column, label in enumerate(LABELS):
        truth, label_probabilities = y_true[:, column], probabilities[:, column]
        predicted = label_probabilities >= THRESHOLD
        tp, fp, fn = (predicted & (truth == 1)).sum(), (predicted & (truth == 0)).sum(), (~predicted & (truth == 1)).sum()
        scores[label] = {
            "AUC": roc_auc_score(truth, label_probabilities),
            "F1": 2 * tp / max(2 * tp + fp + fn, 1), "recall": tp / max(tp + fn, 1), "precision": tp / max(tp + fp, 1),
            "ECE": expected_calibration_error(truth, label_probabilities),
            "Brier": np.mean((label_probabilities - truth) ** 2),
        }
    per_label = pd.DataFrame(scores).T

    resampled_macro_auc = [
        np.mean([roc_auc_score(y_true[rows, column], probabilities[rows, column]) for column in range(len(LABELS))])
        for rows in bootstrap_index
    ]
    auc_low, auc_high = np.percentile(resampled_macro_auc, [2.5, 97.5])
    coverage, automated_accuracy = automation_coverage(y_true[:, 0], probabilities[:, 0])
    summary = {
        "macro AUC": per_label["AUC"].mean(), "macro AUC 95% CI": f"{auc_low:.3f}-{auc_high:.3f}",
        "macro F1": per_label["F1"].mean(), "macro ECE": per_label["ECE"].mean(), "macro Brier": per_label["Brier"].mean(),
        f"toxic: automated at {AUTOMATION_TARGET_ACCURACY:.0%}": coverage, "toxic: accuracy when automated": automated_accuracy,
    }
    return summary, per_label

model_probabilities = {"TF-IDF + LR (160k labelled)": baseline_probabilities}
summaries, per_label_scores = {}, {}
summaries["TF-IDF + LR (160k labelled)"], per_label_scores["TF-IDF + LR (160k labelled)"] = score_probabilities(baseline_probabilities)
display(pd.DataFrame(summaries).T)
per_label_scores["TF-IDF + LR (160k labelled)"].round(3)

## 5. One task, two native formats

Every model sees the same task description and the same six label definitions, word for word.

- **LLMs:** one prompt with the definitions, then the comment, asking for a JSON object with a 0-100 probability per label. Anything that isn't valid JSON with all six keys is logged as invalid.
- **Jev:** the comment is the `state`; the six labels are six `Noul` (yes/no) questions in one call, each with the definition as its criteria. Jev returns six probabilities. Adding questions doesn't add latency, and output tokens are free.

Neither format tells a model which mistakes matter more.

In [ ]:
import json
import re

from typesafe_sdk import Noul, NoulCriteria

TASK_DESCRIPTION = (
    "You are reviewing comments posted on Wikipedia talk pages. For the comment below, judge each of the six labels "
    "independently. A comment can have several labels or none."
)
LABEL_DEFINITIONS = {
    "toxic": "Rude, disrespectful or unreasonable language likely to make someone leave a discussion.",
    "severe_toxic": "Very hateful, aggressive or disrespectful language; an extreme form of toxic.",
    "obscene": "Swear words, vulgar or sexually explicit language.",
    "threat": "An intention to inflict pain, injury or violence on someone.",
    "insult": "Insulting, inflammatory or negative language aimed at a person or group.",
    "identity_hate": "Hateful language aimed at someone for their identity: race, religion, gender, sexuality, disability or nationality.",
}
assert list(LABEL_DEFINITIONS) == LABELS
DEFINITIONS_TEXT = "\n".join(f"- {label}: {definition}" for label, definition in LABEL_DEFINITIONS.items())
LLM_ANSWER_FORMAT = (
    "For each label, give the probability from 0 to 100 that it applies. Respond with only a JSON object with exactly "
    "these six keys and integer values, like " + json.dumps({label: 0 for label in LABELS}) + "."
)

def build_llm_prompt(comment_text):
    return (f"{TASK_DESCRIPTION}\n\n## Labels\n{DEFINITIONS_TEXT}\n\n{LLM_ANSWER_FORMAT}\n\n"
            f"## Comment\n<comment>\n{comment_text}\n</comment>\n\nJSON answer:")

def build_jev_request(comment_text):
    return {
        "state": {"task": TASK_DESCRIPTION, "comment": comment_text},
        "questions": {
            label: Noul(instructions=f"Is this comment {label.replace('_', ' ')}?",
                        criteria=NoulCriteria(true=definition, false=f"The comment is not {label.replace('_', ' ')}."))
            for label, definition in LABEL_DEFINITIONS.items()
        },
    }

def parse_llm_answer(response_text):
    # The outermost {...} must parse as JSON with all six labels as numbers from 0 to 100. Else None (invalid).
    match = re.search(r"\{.*\}", response_text or "", flags=re.DOTALL)
    if not match:
        return None
    try:
        answer = json.loads(match.group(0))
        values = [float(answer[label]) for label in LABELS]
    except (ValueError, KeyError, TypeError):
        return None
    if not all(0 <= value <= 100 for value in values):
        return None
    return [value / 100 for value in values]

example_comment = "You are a complete idiot and nobody wants your edits here."
print(build_llm_prompt(example_comment))
print("\nParser check:", [parse_llm_answer(text) is not None for text in [
    '{"toxic": 90, "severe_toxic": 10, "obscene": 0, "threat": 0, "insult": 85, "identity_hate": 0}',
    '```json\n{"toxic": 5, "severe_toxic": 0, "obscene": 0, "threat": 0, "insult": 2, "identity_hate": 0}\n```',
    '{"toxic": 90}', "I can't help with that.",
]])

## 6. The models

| Model | Tier | Reasoning (provider default) | Output |
|---|---|---|---|
| Claude Sonnet 5 | mid | adaptive thinking, effort high | JSON, 0-100 per label |
| GPT-6 Sol | mid | effort medium | JSON, 0-100 per label |
| Gemini 3.8 Flash | mid | thinking level medium | JSON, 0-100 per label |
| Jev 1.13 | decision model | six calibrated probabilities, one call | typed, cannot be malformed |

- **Same model IDs, keys and prices as the attrition bench** (prices checked 2026-09-25). Environment variables: `CLAUDE_KEY`, `GPT_KEY`, `GEM_KEY`, `JEV_KEY`.
- **No prompt caching.** The shared prefix is about 300 tokens, below every provider's cache minimum, so it wouldn't be reused anyway.
- **Gemini's safety filter is switched off** (`GEMINI_BLOCK_HARMFUL_INPUT = False`). Its default blocks some toxic inputs, which would be a filter setting rather than a judgement. Flip the flag to measure the default instead.
- **Refusals are answers, not errors.** A refused or blocked comment is logged as invalid and goes to a human, like any unparseable answer. Only network or API errors are retried.

In [ ]:
import os
from dataclasses import dataclass, asdict
from datetime import datetime, timezone

import anthropic
import openai
from google import genai
from google.genai import types as genai_types
from typesafe_sdk import TypeSafeClient

JEV_MODEL = "jev-1.13.0"   # pinned version, not jev-latest
MODELS = {
    "Claude Sonnet 5": {"model_id": "claude-sonnet-5", "key_env": "CLAUDE_KEY", "reasoning": "adaptive thinking, effort high",
                        "price": {"input": 2.00, "output": 10.00}},
    "GPT-6 Sol":       {"model_id": "gpt-6-sol", "key_env": "GPT_KEY", "reasoning": "effort medium",
                        "price": {"input": 2.00, "output": 10.00}},
    "Gemini 3.8 Flash": {"model_id": "gemini-3.8-flash", "key_env": "GEM_KEY", "reasoning": "thinking_level medium",
                         "price": {"input": 0.75, "output": 3.75}},
    "Jev":             {"model_id": JEV_MODEL, "key_env": "JEV_KEY", "reasoning": "n/a (six calibrated probabilities)",
                        "price": {"input": 0.042, "output": 0.00}},
}
MAX_OUTPUT_TOKENS = 16_000   # room for reasoning; the visible answer is a small JSON object
GEMINI_BLOCK_HARMFUL_INPUT = False
GEMINI_SAFETY_CATEGORIES = ["HARM_CATEGORY_HARASSMENT", "HARM_CATEGORY_HATE_SPEECH",
                            "HARM_CATEGORY_SEXUALLY_EXPLICIT", "HARM_CATEGORY_DANGEROUS_CONTENT"]

@dataclass
class CallRecord:
    model: str
    comment_index: int
    comment_id: str
    raw_answer: str = ""
    valid: bool = False                     # False = refused, blocked or unparseable -> goes to a human
    p_toxic: float | None = None
    p_severe_toxic: float | None = None
    p_obscene: float | None = None
    p_threat: float | None = None
    p_insult: float | None = None
    p_identity_hate: float | None = None
    latency_s: float = 0.0
    input_tokens: int = 0
    output_tokens: int = 0                  # includes reasoning tokens (billed as output)
    reasoning_tokens: int = 0
    cost_usd: float = 0.0
    served_model: str = ""                  # version string the provider reports
    error: str = ""                         # API/network failure; these rows are retried and never scored
    timestamp: str = ""

PROBABILITY_COLUMNS = [f"p_{label}" for label in LABELS]

def api_key(model_name, env_name=None, required=True):
    env_name = env_name or MODELS[model_name]["key_env"]
    key = os.environ.get(env_name)
    if not key and os.name == "nt":
        # Set in Windows AFTER Jupyter started: the kernel never inherited it, so read the user environment directly
        import winreg
        try:
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as user_environment:
                key = winreg.QueryValueEx(user_environment, env_name)[0]
        except FileNotFoundError:
            key = None
    assert key or not required, f"set {env_name} for {model_name}"
    return key

def call_claude(comment_text, client):
    response = client.messages.create(
        model=MODELS["Claude Sonnet 5"]["model_id"],
        max_tokens=MAX_OUTPUT_TOKENS,
        thinking={"type": "adaptive"},
        output_config={"effort": "high"},
        messages=[{"role": "user", "content": build_llm_prompt(comment_text)}],
    )
    usage = response.usage
    refused = response.stop_reason == "refusal"
    return {
        "raw_answer": "REFUSED" if refused else "".join(block.text for block in response.content if block.type == "text"),
        "input_tokens": usage.input_tokens + (usage.cache_read_input_tokens or 0) + (usage.cache_creation_input_tokens or 0),
        "output_tokens": usage.output_tokens,
        "served_model": response.model,
    }

def call_gpt(comment_text, client):
    response = client.responses.create(
        model=MODELS["GPT-6 Sol"]["model_id"],
        input=build_llm_prompt(comment_text),
        reasoning={"effort": "medium"},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        store=False,
    )
    usage = response.usage
    return {
        "raw_answer": response.output_text or "",
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "reasoning_tokens": usage.output_tokens_details.reasoning_tokens or 0,
        "served_model": response.model,
    }

def call_gemini(comment_text, client):
    safety = None if GEMINI_BLOCK_HARMFUL_INPUT else [
        genai_types.SafetySetting(category=category, threshold="BLOCK_NONE") for category in GEMINI_SAFETY_CATEGORIES
    ]
    response = client.models.generate_content(
        model=MODELS["Gemini 3.8 Flash"]["model_id"],
        contents=build_llm_prompt(comment_text),
        config=genai_types.GenerateContentConfig(
            thinking_config=genai_types.ThinkingConfig(thinking_level="medium"),
            max_output_tokens=MAX_OUTPUT_TOKENS,
            safety_settings=safety,
        ),
    )
    usage = response.usage_metadata
    thoughts = usage.thoughts_token_count or 0
    try:
        text = response.text or ""
    except ValueError:   # a blocked response has no text part
        text = ""
    return {
        "raw_answer": text or "BLOCKED",
        "input_tokens": usage.prompt_token_count or 0,
        "output_tokens": (usage.candidates_token_count or 0) + thoughts,   # thinking is billed as output
        "reasoning_tokens": thoughts,
        "served_model": response.model_version or "",
    }

def call_jev(comment_text, client):
    response = client.system_one(**build_jev_request(comment_text), model=JEV_MODEL)
    probabilities = [response.nouls[label].noul for label in LABELS]
    return {
        "raw_answer": json.dumps({label: round(p, 4) for label, p in zip(LABELS, probabilities)}),
        "probabilities": probabilities,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens or 0,
        "served_model": response.model,
    }

MODEL_CALLERS = {"Claude Sonnet 5": call_claude, "GPT-6 Sol": call_gpt, "Gemini 3.8 Flash": call_gemini, "Jev": call_jev}

def make_client(model_name):
    key = api_key(model_name)
    return {
        # A key not scoped to a workspace needs the workspace ID header (CLAUDE_WORKSPACE_ID, optional otherwise)
        "Claude Sonnet 5": lambda: anthropic.Anthropic(api_key=key, default_headers=(
            {"anthropic-workspace-id": workspace_id}
            if (workspace_id := api_key(model_name, "CLAUDE_WORKSPACE_ID", required=False)) else None
        )),
        "GPT-6 Sol": lambda: openai.OpenAI(api_key=key),
        "Gemini 3.8 Flash": lambda: genai.Client(api_key=key),
        "Jev": lambda: TypeSafeClient(api_key=key, model=JEV_MODEL, timeout=60),
    }[model_name]()

def cost_usd(model_name, fields):
    price = MODELS[model_name]["price"]
    return (fields.get("input_tokens", 0) * price["input"] + fields.get("output_tokens", 0) * price["output"]) / 1_000_000

def predict_one(model_name, comment_index, client):
    # One comment per call. Latency = wall clock around the API call only (prompt building excluded).
    comment = test_set.iloc[comment_index]
    record = CallRecord(model=model_name, comment_index=comment_index, comment_id=comment["id"],
                        timestamp=datetime.now(timezone.utc).isoformat())
    started = time.perf_counter()
    try:
        fields = MODEL_CALLERS[model_name](comment["comment_text"], client)
    except Exception as error:   # logged, never silently dropped; the run loop retries failed rows
        record.latency_s = time.perf_counter() - started
        record.error = f"{type(error).__name__}: {error}"
        return record
    record.latency_s = time.perf_counter() - started
    probabilities = fields.pop("probabilities", None) or parse_llm_answer(fields["raw_answer"])
    for field, value in fields.items():
        setattr(record, field, value)
    record.cost_usd = cost_usd(model_name, fields)
    if probabilities is not None:
        record.valid = True
        for column, probability in zip(PROBABILITY_COLUMNS, probabilities):
            setattr(record, column, probability)
    return record

# Smoke test: ONE call per model on test comment 0, to prove the wiring before the full run (a few cents)
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    smoke_records = [predict_one(model_name, 0, make_client(model_name)) for model_name in MODELS]
    smoke_table = pd.DataFrame([asdict(record) for record in smoke_records]).set_index("model")
    display(smoke_table[["served_model", "valid", *PROBABILITY_COLUMNS, "latency_s", "input_tokens", "output_tokens",
                         "reasoning_tokens", "cost_usd", "error"]])

## 7. The run

1,000 comments x 4 models, **one comment per call**, sequential, the same order for every model. Every call is appended to `llm_call_log.csv` as it returns, so the run survives crashes and resumes where it stopped. The log holds ids, probabilities, tokens and timings, never comment text.

Rough cost at N_TEST = 1,000, from token counts in the attrition run: Claude ~$4-8, GPT ~$2-4, Gemini ~$6-10 (it reasons at length), Jev ~$0.02. Set `RUN_MODELS = ["Jev"]` to start with the cheap one.

In [ ]:
import csv
from dataclasses import fields

CALL_LOG_PATH = Path("llm_call_log.csv")
RUN_MODELS = list(MODELS)   # e.g. ["Jev"] to run one model at a time
MAX_ATTEMPTS = 3
LOG_COLUMNS = [field.name for field in fields(CallRecord)]
TEXT_COLUMNS = ["model", "comment_id", "raw_answer", "valid", "served_model", "error", "timestamp"]

def append_to_log(record, path=CALL_LOG_PATH):
    is_new_log = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as log_file:
        writer = csv.DictWriter(log_file, fieldnames=LOG_COLUMNS)
        if is_new_log:
            writer.writeheader()
        writer.writerow(asdict(record))

def load_call_log(path=CALL_LOG_PATH):
    if not path.exists():
        return pd.DataFrame(columns=LOG_COLUMNS)
    call_log = pd.read_csv(path, dtype=str, keep_default_na=False)
    for column in LOG_COLUMNS:
        if column not in TEXT_COLUMNS:
            call_log[column] = pd.to_numeric(call_log[column], errors="coerce")
    call_log["valid"] = call_log["valid"] == "True"
    return call_log

def succeeded_comment_indexes(model_name):
    call_log = load_call_log()
    return set(call_log.loc[(call_log["model"] == model_name) & (call_log["error"] == ""), "comment_index"].astype(int))

for model_name in RUN_MODELS:
    client = make_client(model_name)
    remaining = [index for index in range(len(test_set)) if index not in succeeded_comment_indexes(model_name)]
    print(f"{model_name}: {len(test_set) - len(remaining)} already done, {len(remaining)} to run")
    for attempt in range(1, MAX_ATTEMPTS + 1):
        failed, last_error = [], ""
        for position, comment_index in enumerate(remaining, start=1):
            record = predict_one(model_name, comment_index, client)
            append_to_log(record)
            if record.error:
                failed.append(comment_index)
                last_error = record.error
            if position % 100 == 0:
                print(f"  {position}/{len(remaining)} calls, {len(failed)} failed so far")
        if not failed:
            break
        print(f"  attempt {attempt}: {len(failed)} failed, last error: {last_error[:120]}")
        remaining = failed
        time.sleep(15 * attempt)

## 8. Results

Every model that answered all 1,000 comments is scored with the same `score_probabilities` as the baseline, next to an efficiency table: invalid answers, latency, cost per 1,000 and per 1M comments.

In [ ]:
call_log = load_call_log()
successful_calls = call_log[call_log["error"] == ""].drop_duplicates(["model", "comment_index"], keep="first")
efficiency = {"TF-IDF + LR (160k labelled)": {
    "invalid answers": 0, "median latency s": baseline_latency_per_comment, "p90 latency s": baseline_latency_per_comment,
    "cost per 1k comments $": 0.0, "cost per 1M comments $": 0.0,
}}
for model_name in MODELS:
    model_calls = successful_calls[successful_calls["model"] == model_name].sort_values("comment_index")
    if len(model_calls) < len(test_set):
        print(f"{model_name}: only {len(model_calls)}/{len(test_set)} answered, not scored yet")
        continue
    assert (model_calls["comment_id"].values == test_set["id"].values).all()
    # Invalid answers (refused, blocked, unparseable) = 0.5 on every label: unknown, never automated
    probabilities = model_calls[PROBABILITY_COLUMNS].astype(float).fillna(0.5).to_numpy()
    model_probabilities[model_name] = probabilities
    summaries[model_name], per_label_scores[model_name] = score_probabilities(probabilities)

    # Cost recomputed from the logged tokens with the CURRENT price table, so a price correction applies retroactively
    call_costs = np.array([cost_usd(model_name, row) for row in model_calls.to_dict("records")])
    efficiency[model_name] = {
        "invalid answers": int((~model_calls["valid"]).sum()),
        "failed attempts": int(((call_log["model"] == model_name) & (call_log["error"] != "")).sum()),
        "median latency s": model_calls["latency_s"].median(),
        "p90 latency s": model_calls["latency_s"].quantile(0.9),
        "cost per 1k comments $": call_costs.mean() * 1_000,
        "cost per 1M comments $": call_costs.mean() * 1_000_000,
        "input tokens / call": model_calls["input_tokens"].mean(),
        "output tokens / call": model_calls["output_tokens"].mean(),
        "reasoning tokens / call": model_calls["reasoning_tokens"].mean(),
        "served model": ", ".join(sorted(model_calls["served_model"].unique())),
    }

summary_table = pd.DataFrame(summaries).T
efficiency_table = pd.DataFrame(efficiency).T
display(summary_table)
display(efficiency_table)
display(pd.concat({model: scores["AUC"] for model, scores in per_label_scores.items()}, axis=1).T.round(3)
        .style.set_caption("ROC AUC per label"))
display(pd.concat({model: scores["ECE"] for model, scores in per_label_scores.items()}, axis=1).T.round(3)
        .style.set_caption("Calibration error (ECE) per label, lower is better"))

### What "automated at 98%" means in practice

For the `toxic` decision, each model's comments are sorted by confidence. The automated share is decided with no human; everything else is queued for review. At 10,000 comments a day, the queue a moderation team has to read is:

In [ ]:
COMMENTS_PER_DAY = 10_000
automation_key = f"toxic: automated at {AUTOMATION_TARGET_ACCURACY:.0%}"
review_queue = pd.DataFrame({
    "automated share": summary_table[automation_key],
    "accuracy when automated": summary_table["toxic: accuracy when automated"],
    f"comments/day for a human (of {COMMENTS_PER_DAY:,})": ((1 - summary_table[automation_key].astype(float)) * COMMENTS_PER_DAY).round(),
    "model cost per day $": efficiency_table["cost per 1k comments $"].astype(float) * COMMENTS_PER_DAY / 1_000,
})
review_queue

## 9. Throughput (optional)

The main run is sequential so latency is clean. A real queue runs calls in parallel. This sends the first `THROUGHPUT_COMMENTS` comments through `THROUGHPUT_WORKERS` threads per model and measures comments per second, logged to `throughput_log.csv`. Provider rate limits apply, so it measures your account tier as much as the model. Off by default.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

RUN_THROUGHPUT_TEST = False
THROUGHPUT_MODELS = ["Jev", "GPT-6 Sol"]
THROUGHPUT_COMMENTS = 200
THROUGHPUT_WORKERS = 16
THROUGHPUT_LOG_PATH = Path("throughput_log.csv")

if RUN_THROUGHPUT_TEST:
    throughput = {}
    for model_name in THROUGHPUT_MODELS:
        client = make_client(model_name)   # the SDK clients are thread-safe for concurrent requests
        started = time.perf_counter()
        with ThreadPoolExecutor(max_workers=THROUGHPUT_WORKERS) as pool:
            records = list(pool.map(lambda index: predict_one(model_name, index, client), range(THROUGHPUT_COMMENTS)))
        wall_seconds = time.perf_counter() - started
        for record in records:
            append_to_log(record, THROUGHPUT_LOG_PATH)
        throughput[model_name] = {
            "workers": THROUGHPUT_WORKERS, "comments": THROUGHPUT_COMMENTS, "wall s": wall_seconds,
            "comments / s": THROUGHPUT_COMMENTS / wall_seconds,
            "errors": sum(bool(record.error) for record in records),
            "hours for 1M comments": 1_000_000 / (THROUGHPUT_COMMENTS / wall_seconds) / 3600,
        }
    display(pd.DataFrame(throughput).T)

## Caveats

- **Jigsaw is public and old (2018).** The comments are almost certainly in every LLM's training data, and possibly Jev's. Labels were never shown in a prompt, but a model may have seen them. Treat the zero-shot scores as an upper bound. A fresh, never-published moderation set would settle it.
- **The test set is enriched to 50% flagged.** Real feeds run nearer 10%. AUC and calibration error are about ranking and honesty per comment, so they carry over; precision and the automated share would shift at a live base rate.
- **The labels are noisy.** Jigsaw labels come from crowd raters, and the definitions used here are one-line paraphrases. Every model faces the same noise and the same definitions.
- **LLM probabilities are verbal.** Asking an LLM for "0-100" is common practice but isn't what they're trained for. That's part of the comparison, since it's how you'd get a confidence out of them today.
- **The baseline trained on 160k labelled comments; nobody else saw any.** It measures what a labelled dataset buys you, not a fair zero-shot competitor.
- **Reasoning at provider defaults** means out of the box, not each model at its best. Levels aren't equivalent across vendors.
- **Latency is measured one sequential call at a time** from one connection. Section 9 measures parallel throughput separately.

## Figures

Every chart in one style, each saved as a PNG in `figures/` (200 dpi). Colour follows the model on every chart: the baseline grey, Jev blue, and the same LLM colours as the attrition bench.

- **01** test-set composition
- **02-03** accuracy: macro AUC with CI, AUC per label
- **04-05** trust: calibration curves, automation coverage
- **06-07** cost and speed: latency, cost per 1M comments
- **08** invalid answers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)
INK, MUTED_INK, GRID, SURFACE = "#0b0b0b", "#52514e", "#e5e4df", "#ffffff"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED_INK, "xtick.color": MUTED_INK, "ytick.color": MUTED_INK,
    "text.color": INK, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "legend.frameon": False,
})
MODEL_COLORS = {
    "TF-IDF + LR (160k labelled)": "#9e9d98",
    "Jev": "#2a78d6", "Claude Sonnet 5": "#eb6834", "GPT-6 Sol": "#1baf7a", "Gemini 3.8 Flash": "#eda100",
}
DISPLAY_NAMES = {"TF-IDF + LR (160k labelled)": "TF-IDF + LR"}
MODEL_ORDER = [model for model in MODEL_COLORS if model in summary_table.index]

def display_name(model_name):
    return DISPLAY_NAMES.get(model_name, model_name)

def style_value_axis(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

def save_figure(fig, name):
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"saved {path}")

# 1. Test-set composition: positives per label
positives = test_set[LABELS].sum()[::-1]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(positives.index, positives.values, height=0.55, color=MODEL_COLORS["Jev"])
for y, count in enumerate(positives.values):
    ax.text(count + 5, y, f"{count}", va="center", color=INK)
ax.set_title(f"Test set: {N_TEST:,} comments, half flagged, every label scorable")
ax.set_xlabel("Comments with this label")
style_value_axis(ax, "x")
save_figure(fig, "01_test_set_labels")

# 2. Macro AUC with bootstrap CI
fig, ax = plt.subplots(figsize=(8, 3.6))
for y, model in enumerate(MODEL_ORDER[::-1]):
    low, high = map(float, summary_table.loc[model, "macro AUC 95% CI"].split("-"))
    value = summary_table.loc[model, "macro AUC"]
    ax.plot([low, high], [y, y], color=MODEL_COLORS[model], linewidth=2.5, solid_capstyle="round")
    ax.plot(value, y, "o", color=MODEL_COLORS[model], markersize=9)
    ax.text(high + 0.004, y, f"{value:.3f}", va="center", color=INK)
ax.set_yticks(range(len(MODEL_ORDER)), [display_name(model) for model in MODEL_ORDER[::-1]])
ax.set_title("Ranking quality: macro ROC AUC over six labels")
ax.set_xlabel("Macro AUC (dot) with 95% bootstrap interval (line)")
style_value_axis(ax, "x")
save_figure(fig, "02_macro_auc")

# 3. AUC per label, grouped
auc_per_label = pd.concat({model: per_label_scores[model]["AUC"] for model in MODEL_ORDER}, axis=1)
fig, ax = plt.subplots(figsize=(11, 4.2))
bar_width = 0.8 / len(MODEL_ORDER)
for offset, model in enumerate(MODEL_ORDER):
    ax.bar(np.arange(len(LABELS)) + offset * bar_width - 0.4 + bar_width / 2, auc_per_label[model],
           width=bar_width, color=MODEL_COLORS[model], label=display_name(model))
ax.set_xticks(range(len(LABELS)), LABELS)
ax.set_ylim(max(0.5, auc_per_label.min().min() - 0.05), 1.0)
ax.set_title("ROC AUC per label")
ax.set_ylabel("AUC")
ax.legend(ncol=len(MODEL_ORDER), loc="lower center", bbox_to_anchor=(0.5, -0.28))
style_value_axis(ax)
save_figure(fig, "03_auc_per_label")

# 4. Calibration curves on all six labels pooled: does a stated 0.8 happen 80% of the time?
fig, ax = plt.subplots(figsize=(6.2, 5.6))
ax.plot([0, 1], [0, 1], color=MUTED_INK, linewidth=1, linestyle="--", label="perfect calibration")
bin_edges = np.linspace(0, 1, CALIBRATION_BINS + 1)
for model in MODEL_ORDER:
    stated, observed = model_probabilities[model].ravel(), y_true.ravel()
    bin_of = np.minimum((stated * CALIBRATION_BINS).astype(int), CALIBRATION_BINS - 1)
    points = [(stated[bin_of == b].mean(), observed[bin_of == b].mean()) for b in range(CALIBRATION_BINS) if (bin_of == b).sum() >= 10]
    ax.plot(*zip(*points), "o-", color=MODEL_COLORS[model], linewidth=2, markersize=5,
            label=f"{display_name(model)} (ECE {summary_table.loc[model, 'macro ECE']:.3f})")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_title("Calibration: stated vs. observed")
ax.set_xlabel("Stated probability (binned)")
ax.set_ylabel("Share that actually had the label")
ax.legend(loc="upper left", fontsize=9)
style_value_axis(ax, "both")
save_figure(fig, "04_calibration")

# 5. Automation coverage curve on the toxic label: accuracy of the automated share as more is automated
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for model in MODEL_ORDER:
    probabilities, truth = model_probabilities[model][:, 0], y_true[:, 0]
    confidence, correct = np.abs(probabilities - 0.5), (probabilities >= THRESHOLD) == truth
    coverage_points, accuracy_points = [], []
    for cutoff in np.unique(confidence)[::-1]:
        if cutoff == 0:
            break
        automated = confidence >= cutoff
        coverage_points.append(automated.mean())
        accuracy_points.append(correct[automated].mean())
    ax.step(coverage_points, accuracy_points, where="post", color=MODEL_COLORS[model], linewidth=2, label=display_name(model))
ax.axhline(AUTOMATION_TARGET_ACCURACY, color=MUTED_INK, linewidth=1, linestyle="--")
ax.text(0.01, AUTOMATION_TARGET_ACCURACY + 0.003, f"{AUTOMATION_TARGET_ACCURACY:.0%} target", color=MUTED_INK, fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0.8, 1.005)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.02))
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_title("How much can each model decide alone? (toxic label)")
ax.set_xlabel("Share of comments decided automatically, most confident first")
ax.set_ylabel("Accuracy on the automated share")
ax.legend(loc="lower left")
style_value_axis(ax, "both")
save_figure(fig, "05_automation_coverage")

# 6-7. Latency and cost per 1M comments, API models only (the baseline runs locally)
api_models = [model for model in MODEL_ORDER if model in MODELS]
for figure_name, column, title, label, formatter in [
    ("06_latency_per_comment", "median latency s", "Median latency per comment (one call, all six labels)", "Seconds", lambda v: f"{v:.2f} s"),
    ("07_cost_per_1m_comments", "cost per 1M comments $", "Cost to moderate 1M comments", "US dollars", lambda v: f"${v:,.0f}"),
]:
    values = efficiency_table.loc[api_models, column].astype(float)
    fig, ax = plt.subplots(figsize=(9, 3.9))
    ax.bar([display_name(model) for model in api_models], values, color=[MODEL_COLORS[model] for model in api_models], width=0.55)
    for x, value in enumerate(values):
        ax.text(x, value * 1.02, formatter(value), ha="center", va="bottom", color=INK)
    ax.set_title(title)
    ax.set_ylabel(label)
    style_value_axis(ax)
    save_figure(fig, figure_name)

# 8. Invalid answers: refused, blocked or malformed
invalid = efficiency_table.loc[api_models, "invalid answers"].astype(int)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar([display_name(model) for model in api_models], invalid, color=[MODEL_COLORS[model] for model in api_models], width=0.55)
for x, value in enumerate(invalid):
    ax.text(x, value + max(invalid.max(), 1) * 0.02, f"{value}", ha="center", va="bottom", color=INK)
ax.set_title(f"Answers that couldn't be used (refused, blocked or malformed), of {N_TEST:,}")
ax.set_ylabel("Comments")
ax.set_ylim(0, max(invalid.max(), 1) * 1.2)
style_value_axis(ax)
save_figure(fig, "08_invalid_answers")